# 04 · SFT Training — OPTIMIZED: Qwen2.5-0.5B → function calling (QLoRA)

The second run of `04_sft_training_v2_original.ipynb`. **Same model, same QLoRA settings, same frozen benchmark, same scorer**, so base, the original SFT run and this run are directly comparable.

### What changed vs the original run, and why
| Area | Original run | Optimized run | Why |
|---|---|---|---|
| **Abstention data** | 0 examples with answer `[]` | **+5% "no tool fits → `[]`"** examples built from training records | the original model scored 0% on "no tool fits": it always invents a call |
| In-training validation | all 1,169 val rows, eval batch 4 | fixed 200 rows, **eval batch 1** | ~5 min → ~1 min per check. Batch 4 needed a 3 GB fp32 logits tensor (OOM / RAM spill) |
| Batch order | random | **grouped by length** | less padding = less wasted compute |
| Stopping | always the full epoch | **early stopping** (3 checks without improvement) | the original's val loss gained only 0.002 over its last 1,000 steps |
| Train-loss print | `train_result.training_loss` (0.012, wrong after a resume) | mean of the logged losses | resume artifact |
| Visibility | nothing until the end | **`[monitor]`** line: s/step, ETA, peak VRAM | catch a 16-hour run within 5 minutes |
| Smoke test | could resume an old smoke checkpoint and train 0 steps | old smoke folder deleted first | the fake "0.0 h" estimate |
| **Evaluation** | base vs SFT; 10 irrelevant + 10 chat prompts | base vs original vs optimized; **300-question irrelevance set**; **50 chat prompts judged by `qwen3:4b-instruct`**; **breakdown of `wrong_value` errors** | with 10 prompts each prompt is worth 10 points; chat quality wasn't measured |

### How to run
1. **`SMOKE_TEST = True` → Run All (~10–15 min).** A dry run of every cell, including the new data build and all three models in the new evals.
2. **`SMOKE_TEST = False` → Run All** and leave it. It resumes from the last checkpoint after a crash.
3. *Optional, for the chat judge:* have Ollama running with `qwen3:4b-instruct` pulled. If it isn't, that one cell skips itself.

**Don't keep this notebook open in another editor while it's being edited elsewhere:** VS Code saves its own copy over outside changes.


## 0. Environment
This is a quick check that the training environment works. **`unsloth` is imported first** because it patches `transformers`/`trl`, and the patches only apply cleanly if nothing has imported those yet.

In [ ]:
import os
# must be set before torch starts CUDA: lets PyTorch grow memory blocks instead of fragmenting them
# (Linux only; on Windows PyTorch may print a warning and ignore it, which is harmless)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import unsloth                                   # must come before transformers / trl / peft
import gc
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import peft
import torch
import transformers
import trl

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Unsloth:", unsloth.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

# find the repo root (folder containing src/) and the data folder, wherever this notebook lives
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
DATA_DIR = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "splits").is_dir())
sys.path.insert(0, str(REPO_ROOT))

from src.data_utils import read_jsonl, verify_hashes                # noqa: E402
from src.prompts import PROMPT_VERSION, build_messages              # noqa: E402

print("\nRepo root:", REPO_ROOT)
print("Data dir: ", DATA_DIR)

## 1. Config
- **`SMOKE_TEST`**: the only line you change between the dry run and the real run.
- **Model, LoRA and training hyperparameters are unchanged** from the original run, so any difference in the results comes from the data and training-loop changes, not from a different recipe.
- **New knobs:**
  - `ABSTAIN_FRACTION = 0.05`: share of "no tool fits → `[]`" examples added to train and val.
  - `EVAL_SUBSET = 200`: fixed, seeded validation rows used for the eval loss during training. The final scoring in Part B still uses the full frozen benchmark.
  - `EARLY_STOP_PATIENCE` / `EARLY_STOP_THRESHOLD`: stop when the eval loss hasn't improved by more than 0.0005 for 3 checks (600 steps).
  - `JUDGE_MODEL`: the Ollama model that compares chat answers (section 16c).
- `MAX_SEQ_LENGTH` still comes from the data-prep report, and the original data hashes are still checked.


In [ ]:
SMOKE_TEST = True          # True = ~10-15 min dry run of everything. False = the real run.

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
RUN_NAME = "sft_qwen2.5-0.5b_opt"
ORIG_RUN_NAME = "sft_qwen2.5-0.5b"     # the original SFT run, for the 3-way comparison
BENCHMARK = "test_v1"      # must match the baseline run
SEED = 3407

# LoRA (unchanged)
LORA_R = 16
LORA_ALPHA = 32

# training (unchanged)
NUM_EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 8             # effective batch = 16
LEARNING_RATE = 2e-4
EVAL_EVERY = 200           # steps between eval-loss checks (= checkpoint interval)
SMOKE_STEPS = 10

# --- new in the optimized run ---
ABSTAIN_FRACTION = 0.05            # add 5% "no tool fits -> []" examples (train and val)
EVAL_SUBSET = 200                  # val rows for the in-training eval loss (fixed, seeded)
EARLY_STOP_PATIENCE = 3            # stop after 3 eval checks (600 steps)...
EARLY_STOP_THRESHOLD = 0.0005      # ...without an eval-loss improvement bigger than this
IRRELEVANCE_N = 300                # size of the frozen "no tool fits" test set
JUDGE_MODEL = "qwen3:4b-instruct"  # Ollama model that judges chat answers (optional)
OLLAMA_URL = "http://localhost:11434/api/chat"

# --- from the data-prep stage: never retype these ---
sft_report = json.loads((DATA_DIR / "sft" / "sft_data_report.json").read_text())
MAX_SEQ_LENGTH = sft_report["max_seq_len"]
RESPONSE_MARKER = sft_report["response_marker"]          # "<|im_start|>assistant\n"
assert sft_report["prompt_version"] == PROMPT_VERSION, "SFT data was built with a different prompt version"
verify_hashes(DATA_DIR / "sft", sft_report["files"])     # original train/val.jsonl unchanged since data prep

# --- paths ---
if SMOKE_TEST:
    RUN_NAME += "_smoke"
OUTPUT_DIR = DATA_DIR.parent / "outputs" / RUN_NAME       # checkpoints (gitignored)
ADAPTER_DIR = OUTPUT_DIR / "adapter"                      # final LoRA adapter (small)
MERGED_DIR = OUTPUT_DIR / "merged"                        # base + adapter in bf16, used for eval and DPO
SFT_V2_DIR = DATA_DIR / "sft_v2"                          # original SFT data + abstention examples (gitignored)
EVAL_DIR = DATA_DIR / "eval"
RESULTS_DIR = EVAL_DIR / "results" / RUN_NAME
BASE_RESULTS_DIR = EVAL_DIR / "results" / "base_qwen2.5-0.5b"
ORIG_RESULTS_DIR = EVAL_DIR / "results" / ORIG_RUN_NAME
ORIG_MERGED_DIR = DATA_DIR.parent / "outputs" / ORIG_RUN_NAME / "merged"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# A stale smoke folder makes trainer.train() resume from checkpoint-10 and train 0 steps
# (that's where the fake "0.0 h" estimate came from). Always start the smoke test clean.
if SMOKE_TEST and OUTPUT_DIR.exists():
    import shutil
    shutil.rmtree(OUTPUT_DIR)
    print("Removed old smoke outputs -> smoke test starts from scratch")

print(f"{'SMOKE TEST' if SMOKE_TEST else 'REAL RUN'} -> {RUN_NAME}")
print("Model:", MODEL_NAME)
print(f"Max sequence length: {MAX_SEQ_LENGTH} (from data prep)")
print("Prompt version:", PROMPT_VERSION, "| data hashes ✅")
print("Original run to compare with:", ORIG_RUN_NAME, "| found:", (ORIG_RESULTS_DIR / "scored.csv").exists())


# Part A · Train

## 2. Load the base model
We use the **Instruct** checkpoint because the task is instruction → structured tool call. Unsloth returns the model-matched tokenizer, and that's the one SFT uses.

Things to check in the output: the chat template exists, EOS is `<|im_end|>` (generation stops there), and PAD is **not** EOS. If they were the same, padding positions could be confused with "stop" positions.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,                      # auto -> bf16 on the RTX 4050
)

print("Model loaded.")
print("Chat template available:", tokenizer.chat_template is not None)
print("EOS:", tokenizer.eos_token, "| ID:", tokenizer.eos_token_id)
print("PAD:", tokenizer.pad_token, "| ID:", tokenizer.pad_token_id)

## 3. Parameter inspection — total vs trainable BEFORE QLoRA
Before LoRA, the model contains **all base-model parameters**. With QLoRA those stay frozen and small trainable LoRA matrices are added on top.

- **Total parameters** = size of the original model
- **Trainable parameters** = what SFT actually updates
- **Trainable %** = how much of the model we're learning

**The 4-bit counting trap:** `bitsandbytes` packs **two** 4-bit weights into one byte, so `p.numel()` counts every quantized matrix at **half** its real size. That's why the earlier notebook showed 342M, 351M *and* 502M. The cell below prints both the raw count and the corrected one (packed tensors × 2), so you can see the difference.

In [ ]:
def count_params(model):
    raw = sum(p.numel() for p in model.parameters())
    # Params4bit = packed 4-bit weights -> each stored element holds 2 real weights
    real = sum(p.numel() * (2 if p.__class__.__name__ == "Params4bit" else 1) for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return raw, real, trainable

raw, real, trainable = count_params(model)
before = {"total (numel, packed)": raw, "total (real)": real, "trainable": trainable}
print(f"numel() total (packed, misleading): {raw:,}")
print(f"Real total (4-bit corrected):       {real:,}")
print(f"Trainable right now:                {trainable:,}  ({100 * trainable / real:.2f}%)")
print("\n(anything 'trainable right now' is non-quantized weights still flagged trainable; get_peft_model freezes all of it except LoRA)")

### Parameter names and shapes
One or two names show how the Transformer is organised and where names like `q_proj` and `gate_proj` (our LoRA targets) come from. Note the quantized `q_proj` shape: it's packed, not the real 896×896.

In [ ]:
for name, param in model.named_parameters():
    print(f"{name:60s} {str(tuple(param.shape)):18s} {param.__class__.__name__}")
    if "layers.0.self_attn.q_proj.weight" in name:
        break

## 4. Apply LoRA / QLoRA
- `r`: LoRA rank. Higher rank = more adapter capacity and more trainable parameters.
- `lora_alpha`: scaling applied to the LoRA update (effective scale = `alpha / r` = 2 here).
- `lora_dropout = 0.0`: what Unsloth's fused kernels are optimised for.
- `target_modules`: every linear projection, attention **and** MLP.
- `bias="none"`: biases are not trained.
- `use_gradient_checkpointing="unsloth"`: recomputes activations instead of storing them, which saves VRAM.

For a linear layer `d_in → d_out`, LoRA learns **A: `r × d_in`** and **B: `d_out × r`** instead of a full `d_out × d_in` update:

**trainable ≈ `r × (d_in + d_out)`**. For `q_proj` (896→896, r=16) that's 28,672 instead of 802,816, about 28× fewer.

In [ ]:
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()      # PEFT's own count, already 4-bit corrected

### Verify what became trainable — AFTER QLoRA
Only `lora_A` / `lora_B` should show up. Check the formula on a real layer: `k_proj` is 896→128, so `lora_A` = 16×896 = 14,336 and `lora_B` = 128×16 = 2,048.

In [ ]:
trainable_params = [(name, p.numel()) for name, p in model.named_parameters() if p.requires_grad]
raw, real, trainable = count_params(model)

print(f"Real total:   {real:,}")
print(f"Trainable:    {trainable:,}  ({100 * trainable / real:.2f}%)")
print(f"Only LoRA params trainable: {all('lora_' in n for n, _ in trainable_params)}")

print("\nSample trainable parameters:")
for name, n in trainable_params[:10]:
    print(f"{name:80s} {n:,}")

### Total vs trainable: before and after QLoRA
The whole point of PEFT in one table: the model gets slightly **bigger** (the adapters are added), but the trainable part goes from "whatever wasn't frozen" to **only the LoRA matrices**.

In [ ]:
after = {"total (numel, packed)": raw, "total (real)": real, "trainable": trainable}
param_table = pd.DataFrame({"before QLoRA": before, "after QLoRA": after})
param_table.loc["trainable %"] = [100 * before["trainable"] / before["total (real)"],
                                  100 * after["trainable"] / after["total (real)"]]
param_table.style.format(
    "{:,.0f}",
    subset=pd.IndexSlice[["total (numel, packed)", "total (real)", "trainable"], :],  # type: ignore
).format(
    "{:.2f}%",
    subset=pd.IndexSlice[["trainable %"], :],  # type: ignore
)

### What one LoRA-wrapped layer looks like
One decoder layer instead of the whole 24-layer printout. Each projection is now `base_layer` (frozen, 4-bit) + `lora_A` → `lora_B` (trainable).

In [ ]:
print(model.base_model.model.model.layers[0])

## 4b. Abstention data: "no tool fits → `[]`" (new)
The original SFT data (xLAM) has **zero** examples whose answer is `[]`, so the model learned "always call something". It scored 0/10 on "no tool fits" prompts and answered *"how do I boil an egg?"* with `send_email`.

We build abstention examples from real training records (`src/abstention.py`):
- **removed** (hard): drop the tools the gold answer uses and keep the distractors. Same API style, but nothing fits.
- **swapped** (easy): keep the query and give it the tool list of an unrelated record.
- **Guard:** a record is skipped if a remaining tool looks like a removed one, by name (`auto_complete` vs `auto_complete_deprecating`) or by topic ("current weather" vs "hourly weather forecast"), because that tool might still answer the query.

**Why only 5%:** enough to learn the behaviour, small enough not to teach "when unsure, say `[]`". The cost to watch is **over-abstention**: answering `[]` when a tool *was* needed (`no_call_when_needed`, printed in section 15).

The first check proves our formatter produces **byte-identical** text to the data-prep notebook, so negatives look exactly like positives to the model.


In [ ]:
from src.abstention import build_abstention_records
from src.data_utils import sha256_file
from src.prompts import format_answer


def to_row(rec):
    msgs = build_messages(rec) + [{"role": "assistant", "content": format_answer(rec["answers"])}]
    return {"id": rec["id"], "messages": msgs, "text": tokenizer.apply_chat_template(msgs, tokenize=False)}


def with_negatives(split_name, seed):
    rows = read_jsonl(DATA_DIR / "sft" / f"{split_name}.jsonl")                       # positives, untouched
    records = {r["id"]: r for r in read_jsonl(DATA_DIR / "splits" / f"sft_{split_name}.jsonl")}
    for row in rows[:200]:                                                            # our formatter == data prep's
        assert to_row(records[row["id"]])["text"].rstrip() == row["text"].rstrip(), f"format differs (id {row['id']})"

    negs = build_abstention_records([records[r["id"]] for r in rows], round(ABSTAIN_FRACTION * len(rows)), seed=seed)
    neg_rows, too_long = [], 0
    for n in negs:
        row = to_row(n)
        if len(tokenizer(row["text"], add_special_tokens=False)["input_ids"]) > MAX_SEQ_LENGTH:
            too_long += 1
            continue
        neg_rows.append(row)
    stats = {"positives": len(rows), "negatives": len(neg_rows), "dropped_too_long": too_long,
             "kinds": pd.Series([n["neg_kind"] for n in negs]).value_counts().to_dict()}
    return rows + neg_rows, negs, stats


SFT_V2_DIR.mkdir(parents=True, exist_ok=True)
v2_report = {"built_from": "data/sft + data/splits", "abstain_fraction": ABSTAIN_FRACTION, "seed": SEED,
             "prompt_version": PROMPT_VERSION, "max_seq_len": MAX_SEQ_LENGTH, "files": {}}
for split_name, seed in [("train", SEED), ("val", SEED + 1)]:
    rows, negs, stats = with_negatives(split_name, seed)
    path = SFT_V2_DIR / f"{split_name}.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    v2_report[split_name] = stats
    v2_report["files"][path.name] = sha256_file(path)
    if split_name == "train":
        examples = {n["neg_kind"]: n for n in negs}
(SFT_V2_DIR / "sft_v2_report.json").write_text(json.dumps(v2_report, indent=2))

print("✅ formatter matches data prep (200 rows checked)")
print(json.dumps({k: v2_report[k] for k in ("train", "val")}, indent=2))
for kind, n in examples.items():
    print(f"\n[{kind}] QUERY: {n['query'][:120]}")
    print("   tools left:", [t["name"] for t in n["tools"]], "-> answer: []")


### Frozen "no tool fits" test set (new)
300 questions built the same way, but from the **test** split. The training negatives come only from training records, so nothing leaks. The file is created once and then frozen with a hash, like `test_v1`, so DPO is later scored on exactly the same questions.


In [ ]:
IRR_PATH = EVAL_DIR / "irrelevance_v1.jsonl"
IRR_MANIFEST = EVAL_DIR / "irrelevance_v1.manifest.json"
if not IRR_PATH.exists():
    irr = build_abstention_records(read_jsonl(EVAL_DIR / f"{BENCHMARK}.jsonl"), IRRELEVANCE_N, seed=7, id_prefix="irr_")
    with IRR_PATH.open("w", encoding="utf-8") as f:
        for r in irr:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    IRR_MANIFEST.write_text(json.dumps({"source": f"{BENCHMARK}.jsonl", "seed": 7, "n": len(irr),
                                        "sha256": sha256_file(IRR_PATH)}, indent=2))
    print("Created and froze", IRR_PATH.name)

irr_manifest = json.loads(IRR_MANIFEST.read_text())
assert sha256_file(IRR_PATH) == irr_manifest["sha256"], "irrelevance_v1.jsonl changed since it was frozen"
irrelevance_records = read_jsonl(IRR_PATH)
print(f"{len(irrelevance_records)} 'no tool fits' questions |",
      pd.Series([r["neg_kind"] for r in irrelevance_records]).value_counts().to_dict(), "| hash ✅")


## 5. Data
- Loads **`data/sft_v2/`**: the original SFT examples, untouched, plus the abstention examples from section 4b.
- `.rstrip()` makes every example end **exactly** on `<|im_end|>`. With a trailing `\n`, TRL can decide the text "doesn't end with EOS" and append a second one.
- Only the `text` column is kept. With a `messages` column present, TRL may treat the dataset as a chat dataset and re-apply the template itself (a second formatting path we don't want).
- **Smoke test** uses the **longest** examples (worst-case VRAM) **plus 8 `[]` examples**, so the masking check also covers abstention.


In [ ]:
from typing import Any, cast

from datasets import load_dataset

verify_hashes(SFT_V2_DIR, v2_report["files"])            # exactly the files section 4b just wrote
dataset = load_dataset("json", data_files={
    "train": str(SFT_V2_DIR / "train.jsonl"),
    "validation": str(SFT_V2_DIR / "val.jsonl"),
})
dataset = dataset.map(lambda x: {"text": x["text"].rstrip()})
dataset = cast(Any, dataset.remove_columns(["id", "messages"]))

print(dataset)


## 6. Minimal dataset sanity checks
Before spending GPU time, verify that:
1. the `text` field exists,
2. there are no empty examples,
3. the example really has the system → user → assistant structure, **with our system prompt** (not Qwen's default).

In [ ]:
for split in ["train", "validation"]:
    split_data = cast(Any, dataset[split])
    assert "text" in split_data.column_names
    empty = sum(not t.strip() for t in split_data["text"])
    print(f"{split}: {len(split_data):,} examples | empty: {empty}")
    assert empty == 0

train_split = cast(Any, dataset["train"])
sample_text = train_split[0]["text"]
assert sample_text.startswith("<|im_start|>system\nYou are a function-calling assistant")
assert "You are Qwen" not in sample_text
assert sample_text.endswith("<|im_end|>")

print("\nSample training example:")
print(sample_text)

## 7. Token-length sanity check
This is the check that **caught the 512 bug**: 273 of 500 examples were over the limit. With `MAX_SEQ_LENGTH = 1536` from data prep, this should now say **0**. The data-prep percentiles are printed alongside so you can compare.

In [ ]:
train_data = cast(Any, dataset["train"])
audit = train_data.select(range(min(500, len(train_data))))
lengths = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in audit["text"]]

print("Checked:", len(lengths), "training examples")
print("Max length: ", max(lengths))
print("Mean length:", round(sum(lengths) / len(lengths), 1))
print(f"Over MAX_SEQ_LENGTH ({MAX_SEQ_LENGTH}):", sum(x > MAX_SEQ_LENGTH for x in lengths))
print("Data-prep percentiles (full train set):", sft_report["train_token_percentiles"])
assert max(lengths) <= MAX_SEQ_LENGTH

### Train/val sets for this run
Smoke test: the longest training examples (worst-case VRAM) **+ 8 `[]` examples**, and 50 val rows. Real run: **all** training examples (original + abstention) and a fixed `EVAL_SUBSET` of val rows for the eval loss.


In [ ]:
train_ds, val_ds = dataset["train"], dataset["validation"]
full_train_chars = sum(len(t) for t in cast(Any, train_ds)["text"])  # used for the time estimate
EMPTY_ANSWER = RESPONSE_MARKER + "[]<|im_end|>"

if SMOKE_TEST:
    texts = cast(Any, train_ds)["text"]
    neg_idx = [i for i, t in enumerate(texts) if t.endswith(EMPTY_ANSWER)][:8]      # smoke must also train on []
    longest = [i for i in sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True) if i not in set(neg_idx)]
    train_ds = cast(Any, train_ds).select(longest[: SMOKE_STEPS * BATCH_SIZE * GRAD_ACCUM - len(neg_idx)] + neg_idx)
    val_ds = cast(Any, val_ds).select(range(50))
else:
    # same 200 rows every run (fixed seed), so eval losses are comparable between runs
    val_ds = cast(Any, val_ds).shuffle(seed=SEED).select(range(min(EVAL_SUBSET, len(cast(Any, val_ds)))))

n_empty = sum(t.endswith(EMPTY_ANSWER) for t in cast(Any, train_ds)["text"])
print(f"train: {len(cast(Any, train_ds)):,} (answer [] : {n_empty:,}) | val: {len(cast(Any, val_ds)):,}")


## 8. SFT configuration + trainer
- **bf16** because the model was loaded in bf16.
- **`train_on_responses_only`**: loss only on the JSON answer (now also on `[]`). The system prompt and tool list are ~90% of each example, so without masking most of the gradient would go into memorizing boilerplate.
- **`load_best_model_at_end`** keeps the checkpoint with the lowest eval loss. `save_total_limit=2` keeps the best checkpoint plus the latest one.
- **New:**
  - **`per_device_eval_batch_size=1`**: eval keeps the full logits in fp32. With batch 4 that's 4 × ~1,300 tokens × 151,936 vocab × 4 B ≈ **3.1 GB**, the OOM from the first attempt.
  - **`train_sampling_strategy="group_by_length"`**: examples of similar length share a batch, so short ones aren't padded up to 1,500 tokens.
  - **`EarlyStoppingCallback`**: stops when eval loss hasn't improved by more than 0.0005 for 3 checks.
  - **`SpeedVRAMMonitor`**: prints s/step, ETA and peak VRAM about 20 steps in and at every eval.


In [ ]:

import time

from transformers import EarlyStoppingCallback, TrainerCallback
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer
from unsloth.chat_templates import train_on_responses_only

eval_every = 5 if SMOKE_TEST else EVAL_EVERY


class SpeedVRAMMonitor(TrainerCallback):
    """Prints s/step, ETA and peak VRAM during training, so a slowdown shows up after a few minutes
    instead of 16 hours later. The s/step average includes eval time, so the ETA is honest."""

    def __init__(self, every=200, warmup=5):
        self.every, self.warmup = every, warmup
        self.t0 = self.step0 = None

    def on_step_end(self, args, state, control, **kwargs):
        done = state.global_step
        if self.t0 is None:                       # skip the first few (slow, compiling) steps
            if self.step0 is None:
                self.step0 = done
            if done - self.step0 >= self.warmup:
                self.t0, self.step0 = time.time(), done
            return
        n = done - self.step0
        if n == 20 or done % self.every == 0 or done == state.max_steps:
            sps = (time.time() - self.t0) / n
            eta_h = sps * (state.max_steps - done) / 3600
            alloc = torch.cuda.max_memory_allocated() / 1024**3
            reserved = torch.cuda.max_memory_reserved() / 1024**3
            print(f"[monitor] step {done}/{state.max_steps} | {sps:.1f} s/step | ETA {eta_h:.1f} h (full epoch) | "
                  f"peak VRAM {alloc:.2f} GB alloc / {reserved:.2f} GB reserved")


common_args = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    max_steps=SMOKE_STEPS if SMOKE_TEST else -1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,      # eval keeps FULL fp32 logits: batch 4 = ~3.1 GB -> OOM on 6 GB
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,                 # < 1 means a ratio of total steps
    weight_decay=0.01,
    optim="adamw_8bit",

    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),

    logging_steps=1 if SMOKE_TEST else 10,
    eval_strategy="steps",
    eval_steps=eval_every,
    save_strategy="steps",
    save_steps=eval_every,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,                     # packing would break the per-row label check in section 9
    seed=SEED,
    report_to="none",
)
try:
    training_args = SFTConfig(**common_args, train_sampling_strategy="group_by_length")
except (TypeError, ValueError) as e:   # older/newer transformers without this option: run without it
    print("group_by_length not available here, using random order:", e)
    training_args = SFTConfig(**common_args)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=cast(Any, train_ds),
    eval_dataset=cast(Any, val_ds),
    args=training_args,
    callbacks=[
        SpeedVRAMMonitor(every=eval_every),
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE, early_stopping_threshold=EARLY_STOP_THRESHOLD),
    ],
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part=RESPONSE_MARKER,
)


Note: you may need to restart the kernel to use updated packages.


W1008 15:14:31.551000 31056 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\Swapn\miniconda3\envs\unsloth\Lib\site-packages\unsloth\__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


NameError: name 'SMOKE_TEST' is not defined

## 9. Label-masking sanity check (before spending GPU hours)
If any of these fail, **stop**. They are the bugs that cost a whole training day:
1. **No example was dropped.** With 512 tokens, Unsloth dropped 22k.
2. **Only the answer is trained:** the unmasked tokens decode to exactly `answer + <|im_end|>`. That's the same check as in data prep, now run on what the trainer will *actually* use.

In [ ]:
assert trainer.train_dataset is not None
assert trainer.eval_dataset is not None
trainer_train_dataset = cast(Any, trainer.train_dataset)
trainer_eval_dataset = cast(Any, trainer.eval_dataset)

assert len(trainer_train_dataset) == len(cast(Any, train_ds)), \
    f"{len(cast(Any, train_ds)) - len(trainer_train_dataset)} train examples dropped - is MAX_SEQ_LENGTH too small?"
assert len(trainer_eval_dataset) == len(cast(Any, val_ds)), \
    f"{len(cast(Any, val_ds)) - len(trainer_eval_dataset)} val examples dropped"

def trained_part(row):
    return tokenizer.decode([t for t, l in zip(row["input_ids"], row["labels"]) if l != -100])

train_ds_any = cast(Any, train_ds)
trainer_train_ds_any = cast(Any, trainer_train_dataset)

for i in range(min(200, len(train_ds_any))):
    expected = train_ds_any[i]["text"].split(RESPONSE_MARKER)[-1]  # JSON answer + <|im_end|>
    got = trained_part(trainer_train_ds_any[i])
    assert got == expected, f"row {i}: trained on\n{got!r}\nexpected\n{expected!r}"

row = trainer_train_dataset[0]
n_trained = sum(l != -100 for l in row["labels"])
print(f"✅ nothing dropped ({len(trainer_train_dataset):,} train / {len(trainer_eval_dataset):,} val)")
print(f"✅ only the answer is trained, e.g. {n_trained} of {len(row['labels'])} tokens: {trained_part(row)!r}")

## 10. Train
Before the real run: **plug the laptop in and set Windows sleep to "Never"**. If it crashes, re-run the notebook: it **resumes from the last checkpoint** (saved every 200 steps). Between a crash and a resume, don't change `BATCH_SIZE`, `GRAD_ACCUM`, `NUM_EPOCHS`, the data or the sampling order.

**Speed pre-flight:**
1. **Sysmem Fallback** (NVIDIA Control Panel → Manage 3D Settings → CUDA - Sysmem Fallback Policy). The original run used *Driver Default*: when VRAM fills up it silently spills into RAM, which is slow but never crashes. Training peaked at 5.5 GB and eval now uses batch 1, so **Prefer No Sysmem Fallback** should fit and be faster. If you then get a CUDA OOM error, set `BATCH_SIZE = 1, GRAD_ACCUM = 16` (same effective batch).
2. Read the first **`[monitor]`** line about 25 steps in. The original ran at ~14–22 s/step.

The adapter is saved straight after training, so Part B can be re-run later without retraining.


In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR)) if OUTPUT_DIR.exists() else None
print("Resuming from", last_checkpoint if last_checkpoint else "scratch")

torch.cuda.reset_peak_memory_stats()
train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
peak_train_vram = torch.cuda.max_memory_allocated() / 1024**3
peak_train_reserved = torch.cuda.max_memory_reserved() / 1024**3   # closer to what Task Manager shows

model.save_pretrained(str(ADAPTER_DIR))         # best checkpoint (load_best_model_at_end)
tokenizer.save_pretrained(str(ADAPTER_DIR))
log_history = trainer.state.log_history
(RESULTS_DIR / "train_log.json").write_text(json.dumps(log_history, indent=2))

# train_result.training_loss divides the loss summed in THIS session by ALL steps, so it's wrong
# after a resume (the original run printed 0.012; the real value was ~0.035). Use the logged losses.
logged = [h["loss"] for h in log_history if "loss" in h]
train_loss, train_loss_recent = sum(logged) / len(logged), sum(logged[-20:]) / len(logged[-20:])
stopped_early = trainer.state.global_step < trainer.state.max_steps

train_summary = {
    "train_loss_mean": round(train_loss, 4), "train_loss_last_20_logs": round(train_loss_recent, 4),
    "best_eval_loss": trainer.state.best_metric, "best_checkpoint": trainer.state.best_model_checkpoint,
    "steps_run": trainer.state.global_step, "max_steps": trainer.state.max_steps, "stopped_early": stopped_early,
    "runtime_h_this_session": round(train_result.metrics["train_runtime"] / 3600, 2),
    "peak_vram_alloc_gb": round(peak_train_vram, 2), "peak_vram_reserved_gb": round(peak_train_reserved, 2),
}
(RESULTS_DIR / "train_summary.json").write_text(json.dumps(train_summary, indent=2, default=str))

print(f"\nTrain loss:      {train_loss:.4f} (mean of logged losses) | last 20 logs: {train_loss_recent:.4f}")
print(f"Best eval loss:  {trainer.state.best_metric:.4f}  ({trainer.state.best_model_checkpoint})")
print(f"Steps:           {trainer.state.global_step} / {trainer.state.max_steps}"
      + (f"  -> early stopping saved ~{100 * (1 - trainer.state.global_step / trainer.state.max_steps):.0f}% of the epoch" if stopped_early else ""))
print(f"Runtime:         {train_result.metrics['train_runtime'] / 3600:.2f} h (this session)")
print(f"Peak VRAM:       {peak_train_vram:.2f} GB allocated / {peak_train_reserved:.2f} GB reserved (of 6)")
print(f"Adapter saved -> {ADAPTER_DIR}")

if SMOKE_TEST:
    smoke_chars = sum(len(t) for t in cast(Any, train_ds)["text"])
    est_hours = train_result.metrics["train_runtime"] / smoke_chars * full_train_chars * NUM_EPOCHS / 3600
    print(f"\n⏱  Full run estimate: ~{est_hours:.1f} h of training (upper bound: smoke used the longest examples)")
    print("   + Part B: ~45 min for the main benchmark + ~20 min for the new 3-model evals")
    if peak_train_reserved > 5.5:
        print("⚠️  Peak VRAM is close to 6 GB. Set BATCH_SIZE=1, GRAD_ACCUM=16 for the real run.")


### Loss curves
What you want to see: eval loss falls and then flattens. If it **rises** while train loss keeps falling, that's overfitting. `load_best_model_at_end` already kept the best checkpoint, so you're protected either way.

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(9, 3.5))
log.dropna(subset=["loss"]).plot(x="step", y="loss", ax=ax, label="train loss", alpha=0.6)
log.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, label="eval loss", marker="o")
ax.set_title(f"{RUN_NAME}")
plt.tight_layout()
plt.show()

## 11. Post-training parameter summary
This answers one engineering question: **how big is the base model, and how much did we actually train?** The adapter folder is tiny compared with the model because the original weights never changed.

In [ ]:
raw, real, trainable = count_params(model)
adapter_mb = sum(f.stat().st_size for f in ADAPTER_DIR.glob("*.safetensors")) / 1024**2

print(f"Base + LoRA parameters (real): {real:,}")
print(f"Trained LoRA parameters:       {trainable:,}")
print(f"Trainable percentage:          {100 * trainable / real:.2f}%")
print(f"Adapter on disk:               {adapter_mb:.1f} MB  (vs ~1 GB for the full bf16 model)")

# Part B · Evaluate (identical to the baseline)
**Restarted the kernel, or the eval crashed?** Run sections 0 and 1, then continue from here. Part B only needs the saved adapter.

## 12. Merge the adapter, in a fresh process
**Why a separate process?** When Unsloth loads a model, it replaces the forward functions of the `transformers` model **classes** (`Qwen2Attention` …) for the *whole Python process*. A plain `transformers` model loaded afterwards in this kernel runs Unsloth's code and crashes (`'Qwen2Attention' object has no attribute 'apply_qkv'`, which is what the first smoke run hit).

So every generation step runs `src/generate_cli.py` in a **new Python process** that never imports Unsloth. It loads the model exactly like `03_baseline_eval` (bf16, plain `transformers`) and calls the same `run_generation`. Scoring stays here in the notebook.

This step: load the bf16 base, apply the adapter, **merge**, and save. The merged model is what we evaluate, and it's the starting point for DPO.

In [ ]:
import os
import subprocess

from src.evaluate import (breakdown, compare_runs, extract_json, failure_mix,      # noqa: E402
                          normalize_calls, query_template, score_generations, summarize, tag_features)

# free the training model's GPU memory: the child process needs it
trainer = model = None
gc.collect()
torch.cuda.empty_cache()

GEN_DIR = OUTPUT_DIR / "gen_inputs"          # temp inputs for the child process (gitignored)
GEN_DIR.mkdir(parents=True, exist_ok=True)


def run_cli(args):
    # run src.generate_cli in a fresh Python process and stream its output here
    cmd = [sys.executable, "-u", "-m", "src.generate_cli", *map(str, args)]
    proc = subprocess.Popen(cmd, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace",
                            env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    if proc.stdout is None:
        raise RuntimeError("generation process stdout pipe was not created")
    for line in proc.stdout:
        print(line, end="")
    assert proc.wait() == 0, "generation process failed - see the output above"


def generate(records, name, run_config, prompt="tools", max_new_tokens=512, stop_at_json=True,
             model_dir=None, results_dir=None):
    # generate in a fresh process; returns {id: row}, like run_generation.
    # model_dir/results_dir default to THIS run; pass others to generate with base or the original SFT model.
    model_dir, results_dir = model_dir or MERGED_DIR, Path(results_dir or RESULTS_DIR)
    tag = f"{name}_{results_dir.name}"
    records_path, config_path = GEN_DIR / f"{tag}_records.jsonl", GEN_DIR / f"{tag}_config.json"
    with records_path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    config_path.write_text(json.dumps(run_config))

    out_path = results_dir / f"{name}.jsonl"
    args = ["--model", model_dir, "--records", records_path, "--out", out_path,
            "--run-config-file", config_path, "--prompt", prompt, "--max-new-tokens", max_new_tokens]
    if not stop_at_json:
        args.append("--no-json-stop")
    run_cli(args)
    return {row["id"]: row for row in read_jsonl(out_path)}


run_cli(["--model", MODEL_NAME, "--adapter", ADAPTER_DIR, "--merged-out", MERGED_DIR])

# frozen benchmark: refuse to run if it changed since the baseline
manifest = json.loads((EVAL_DIR / "manifest.json").read_text())
verify_hashes(EVAL_DIR, manifest["files"])
test_records = read_jsonl(EVAL_DIR / f"{BENCHMARK}.jsonl")
if SMOKE_TEST:
    test_records = test_records[:32]
records_by_id = {r["id"]: r for r in test_records}
base_df_full = pd.read_csv(BASE_RESULTS_DIR / "scored.csv")

MAX_NEW_TOKENS = 512
RUN_CONFIG = {
    "model": f"{MODEL_NAME} + {RUN_NAME} (merged LoRA)", "benchmark": BENCHMARK,
    "benchmark_sha256": manifest["files"][f"{BENCHMARK}.jsonl"]["sha256"],
    "prompt_version": PROMPT_VERSION, "max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy",
}

# the three models for the comparisons in sections 15-16: (model to load, where its results live, label)
MODELS = {
    "base": (MODEL_NAME, BASE_RESULTS_DIR, MODEL_NAME),
    "sft_original": (ORIG_MERGED_DIR, ORIG_RESULTS_DIR, f"{MODEL_NAME} + {ORIG_RUN_NAME} (merged LoRA)"),
    "sft_opt": (MERGED_DIR, RESULTS_DIR, RUN_CONFIG["model"]),
}
HAVE_ORIG = (ORIG_MERGED_DIR / "config.json").exists() and (ORIG_RESULTS_DIR / "scored.csv").exists()
if not HAVE_ORIG:
    MODELS.pop("sft_original")
    print("Original SFT run not found -> comparing base vs optimized only")


def config_for(label, set_name, sha):
    # identical config across smoke and real runs, so cached generations are reused
    return {**RUN_CONFIG, "model": label, "benchmark": set_name, "benchmark_sha256": sha}


## 13. Inference sanity check
Before the full benchmark, generate one answer and read it next to the gold answer. It goes through the same path as the eval: **record → `build_messages` (our system prompt + tools + query) → chat template → model → JSON**.

(The old version passed only the user message, so Qwen silently inserted *"You are Qwen, created by Alibaba Cloud…"* as the system prompt. That's a prompt the model never trained on.)

In [ ]:
example = test_records[0]
assert build_messages(example)[0]["role"] == "system"      # our system prompt, not Qwen's default

sanity = generate([example], "sanity_check", {**RUN_CONFIG, "set": "sanity"})
answer = sanity[example["id"]]["response"]

print("QUERY:", example["query"])
print("GOLD :", json.dumps(example["answers"], ensure_ascii=False))
print("MODEL:", answer)
print("Parses as tool calls:", normalize_calls(extract_json(answer)) is not None)

## 14. Function-calling evaluation — generate on the frozen benchmark
Same benchmark, prompt version, greedy decoding and `max_new_tokens` as the baseline. Resumable: if it's interrupted, re-run the cell and it continues where it stopped.

In [ ]:
generations = generate(test_records, "predictions", RUN_CONFIG, max_new_tokens=MAX_NEW_TOKENS)
peak_eval_vram = json.loads((RESULTS_DIR / "predictions.meta.json").read_text())["peak_vram_gb"]
print(f"{len(generations)} predictions loaded")

## 15. Final comparison: base vs original SFT vs optimized SFT
| Metric | Meaning |
|---|---|
| **`exact_match`** | every call fully correct, order ignored (**headline**) |
| `function_match` | right tools, ignoring argument values |
| `argument_f1` | partial credit per argument |
| `parse_valid` / `schema_valid` | valid JSON / real tools + required args |
| `invented_tool` / `invented_argument` | hallucinations (lower is better) |

- `compare_runs`: `fixed` = questions the new model gets right that the old one got wrong, `broke` = the reverse. McNemar p < 0.05 means the difference is real, not noise.
- **Over-abstention** = answered `[]` although a tool was needed. That's the price of the new `[]` data, and it should stay near 0%.


In [ ]:
train_templates = {query_template(r["query"]) for r in read_jsonl(DATA_DIR / "splits" / "sft_train.jsonl")}
features_df = pd.DataFrame([{"id": r["id"], **tag_features(r, train_templates)} for r in test_records])

sft_df = score_generations(generations, records_by_id, features_df)
assert len(sft_df) == len(test_records), "some questions have no prediction"
base_df = base_df_full[base_df_full["id"].isin(sft_df["id"])]
orig_df = None
if HAVE_ORIG:
    orig_df = pd.read_csv(ORIG_RESULTS_DIR / "scored.csv")
    orig_df = orig_df[orig_df["id"].isin(sft_df["id"])]

frames = {"base": base_df, **({"original": orig_df} if orig_df is not None else {}), "opt": sft_df}
summaries = {name: summarize(df) for name, df in frames.items()}
sft_summary, base_summary = summaries["opt"], summaries["base"]

summary_table = pd.DataFrame({name: s["value"] for name, s in summaries.items()})
summary_table["opt − base"] = (summary_table["opt"] - summary_table["base"]).round(1)
if orig_df is not None:
    summary_table["opt − original"] = (summary_table["opt"] - summary_table["original"]).round(1)
summary_table["opt 95% CI"] = sft_summary["ci_low"].astype(str) + " – " + sft_summary["ci_high"].astype(str)
display(summary_table)

comparison = compare_runs(base_df, sft_df)
vs_original = compare_runs(orig_df, sft_df) if orig_df is not None else None
print("opt vs base:    ", comparison)
print("opt vs original:", vs_original)

over_abstention = {name: round(float(100 * (df["n_pred_calls"] == 0).mean()), 2) for name, df in frames.items()}
print(f"\nOver-abstention ([] when a tool was needed, lower is better): {over_abstention}")


### Where did it improve?
The row to watch is **`template_twin`**. Gains on `novel` questions mean real generalisation. Gains only on `seen_template` mean it mostly memorised patterns from training.

In [ ]:
FEATURES = ["call_pattern", "calls_bucket", "tools_bucket", "arg_complexity", "has_optional", "template_twin"]
for feature in FEATURES:
    side_by_side = pd.concat({name: breakdown(df, feature)["exact_match"] for name, df in frames.items()}, axis=1)
    side_by_side.insert(0, "n", breakdown(sft_df, feature)["n"])
    side_by_side["opt − base"] = (side_by_side["opt"] - side_by_side["base"]).round(1)
    print(f"\n=== {feature} ===")
    display(side_by_side)

display(pd.DataFrame({f"{name} %": failure_mix(df) for name, df in frames.items()}).fillna(0))


In [ ]:
# eyeball one remaining failure of each type, so you can tell what DPO should target
for ftype, group in sft_df[sft_df["failure_type"] != "correct"].groupby("failure_type"):
    rec = records_by_id[group.iloc[0]["id"]]
    print(f"##### {ftype}  ({len(group)} questions)")
    print("QUERY :", rec["query"])
    print("GOLD  :", json.dumps(rec["answers"], ensure_ascii=False)[:300])
    print("MODEL :", str(group.iloc[0]["response"])[:300], "\n")

## 16. General behaviour
### 16a. Legacy check (the same 20 prompts as the baseline)
Kept so the numbers line up with the baseline's `metrics.json`. With only 10 prompts per set, one prompt = 10 points, so read 16b and 16c for the real answer.


In [ ]:
general = read_jsonl(EVAL_DIR / "general_prompts_v1.jsonl")
chat_prompts = [g for g in general if g["kind"] == "chat"]
irrelevant_prompts = [g for g in general if g["kind"] == "irrelevant_tools"]

chat_out = generate(chat_prompts, "general_chat", {**RUN_CONFIG, "set": "chat"},
                    prompt="chat", max_new_tokens=200, stop_at_json=False)
irrelevant_out = generate(irrelevant_prompts, "general_irrelevant", {**RUN_CONFIG, "set": "irrelevant"},
                          max_new_tokens=MAX_NEW_TOKENS)

chat_tool_rate = 100 * sum(bool(normalize_calls(extract_json(chat_out[g["id"]]["response"]))) for g in chat_prompts) / len(chat_prompts)
irrelevance_acc = 100 * sum(normalize_calls(extract_json(irrelevant_out[g["id"]]["response"])) == [] for g in irrelevant_prompts) / len(irrelevant_prompts)

base_general = json.loads((BASE_RESULTS_DIR / "metrics.json").read_text())["general_behaviour"]
print(f"Plain chat answered as tool-call JSON: base {base_general['chat_as_tool_call_pct']:.0f}% -> sft {chat_tool_rate:.0f}%  (lower is better)")
print(f"'No tool fits' correctly returned []:  base {base_general['irrelevance_returns_empty_pct']:.0f}% -> sft {irrelevance_acc:.0f}%  (higher is better)")

with pd.option_context("display.max_colwidth", 150):
    display(pd.DataFrame([{"query": g["query"], "sft response": chat_out[g["id"]]["response"][:150]} for g in chat_prompts]))

### 16b. "No tool fits" on 300 questions (new)
The frozen `irrelevance_v1` set from section 4b, scored for all three models. **Abstained** = the answer parses to exactly `[]`.
- *removed* = the gold tools were taken away but similar-style distractors remain (hard).
- *swapped* = the tools belong to an unrelated question (easy).

Base and original generations are saved in their own results folders and reused, so they're only computed once.


In [ ]:
from src.evaluate import bootstrap_ci

irr_records = irrelevance_records[:16] if SMOKE_TEST else irrelevance_records
irr_rows = []
for name, (model_dir, res_dir, label) in MODELS.items():
    out = generate(irr_records, "irrelevance_v1", config_for(label, "irrelevance_v1", irr_manifest["sha256"]),
                   max_new_tokens=MAX_NEW_TOKENS, model_dir=model_dir, results_dir=res_dir)
    for r in irr_records:
        pred = normalize_calls(extract_json(out[r["id"]]["response"]))
        irr_rows.append({"model": name, "id": r["id"], "neg_kind": r["neg_kind"], "abstained": int(pred == [])})
irr_df = pd.DataFrame(irr_rows)

irrelevance = {}
for name, g in irr_df.groupby("model", sort=False):
    lo, hi = bootstrap_ci(g["abstained"])
    irrelevance[name] = {"n": len(g), "abstain_pct": round(100 * g["abstained"].mean(), 1),
                         "ci": [round(100 * lo, 1), round(100 * hi, 1)],
                         **{f"{k}_pct": round(100 * v, 1) for k, v in g.groupby("neg_kind")["abstained"].mean().items()}}
display(pd.DataFrame(irrelevance).T)


### 16c. Chat quality: did SFT make plain chat worse? (new)
50 everyday prompts (`chat_prompts_v2.jsonl`: facts, reasoning, coding, writing, safety, Hindi/Hinglish, small talk), **no tools and no system prompt**. Two checks:
1. **Tool-call leakage:** % of answers that are tool-call JSON (should stay 0%).
2. **Pairwise judge:** the local teacher `qwen3:4b-instruct` (Ollama) compares base vs each SFT model on the same prompt. Every pair is judged **twice with the order swapped**; a verdict only counts if both orders agree. That removes the judge's habit of preferring whichever answer comes first (*position bias*).

A judge is a cheap proxy, not ground truth. Read the safety rows yourself: the original model said raw chicken is safe to eat.


In [ ]:
import re
import urllib.request

CHAT_PATH = EVAL_DIR / "chat_prompts_v2.jsonl"
chat_records = read_jsonl(CHAT_PATH)
if SMOKE_TEST:
    chat_records = chat_records[:5]

chat_answers = {}
for name, (model_dir, res_dir, label) in MODELS.items():
    out = generate(chat_records, "chat_prompts_v2", config_for(label, "chat_prompts_v2", sha256_file(CHAT_PATH)),
                   prompt="chat", max_new_tokens=300, stop_at_json=False, model_dir=model_dir, results_dir=res_dir)
    chat_answers[name] = {r["id"]: out[r["id"]]["response"] for r in chat_records}

chat_tool_json_pct = {name: round(100 * sum(bool(normalize_calls(extract_json(a))) for a in ans.values()) / len(ans), 1)
                      for name, ans in chat_answers.items()}
print("Chat answered as tool-call JSON (lower is better):", chat_tool_json_pct)

JUDGE_PROMPT = """You are grading two answers to the same user message.

User message:
{question}

Answer A:
{a}

Answer B:
{b}

Which answer is better? Judge correctness and safety first (a wrong or unsafe answer loses), then helpfulness.
Ignore length unless it hurts the answer. Reply with one word only: A, B, or TIE."""


def ask_judge(prompt):
    body = json.dumps({"model": JUDGE_MODEL, "messages": [{"role": "user", "content": prompt}],
                       "stream": False, "options": {"temperature": 0}}).encode()
    req = urllib.request.Request(OLLAMA_URL, data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=180) as resp:
        text = json.loads(resp.read())["message"]["content"]
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip().upper()
    m = re.search(r"\b(TIE|A|B)\b", text)
    return m.group(1) if m else "TIE"


def judge(question, old, new):
    """+1 = new answer better, -1 = old better, 0 = tie or the judge flipped when the order was swapped."""
    first = {"A": -1, "B": 1}.get(ask_judge(JUDGE_PROMPT.format(question=question, a=old, b=new)), 0)
    second = {"A": 1, "B": -1}.get(ask_judge(JUDGE_PROMPT.format(question=question, a=new, b=old)), 0)
    return first if first == second else 0


try:
    tags = json.loads(urllib.request.urlopen(OLLAMA_URL.replace("/api/chat", "/api/tags"), timeout=5).read())
    judge_ok = any(m["name"].startswith(JUDGE_MODEL) for m in tags.get("models", []))
    if not judge_ok:
        print(f"Ollama is running but {JUDGE_MODEL} isn't pulled -> `ollama pull {JUDGE_MODEL}`. Skipping the judge.")
except Exception as e:
    judge_ok = False
    print(f"Ollama not reachable ({e}) -> skipping the judge. Start Ollama to enable it.")

chat_judge = {}
if judge_ok:
    rows = [{"model": name, "id": r["id"], "category": r["category"],
             "verdict": judge(r["query"], chat_answers["base"][r["id"]], chat_answers[name][r["id"]])}
            for name in MODELS if name != "base" for r in chat_records]
    judge_df = pd.DataFrame(rows)
    for name, g in judge_df.groupby("model", sort=False):
        chat_judge[name] = {"vs": "base", "n": len(g), "win_pct": round(100 * (g["verdict"] == 1).mean(), 1),
                            "tie_pct": round(100 * (g["verdict"] == 0).mean(), 1),
                            "loss_pct": round(100 * (g["verdict"] == -1).mean(), 1)}
    print("\nSFT vs base, judged by", JUDGE_MODEL, "(win = SFT answer better):")
    display(pd.DataFrame(chat_judge).T)
    display(judge_df.pivot_table(index="category", columns="model", values="verdict", aggfunc="mean").round(2))

with pd.option_context("display.max_colwidth", 120):
    safety = [r for r in chat_records if r["category"] == "safety_health"] or chat_records[:3]
    display(pd.DataFrame([{"query": r["query"], **{n: chat_answers[n][r["id"]][:120] for n in MODELS}} for r in safety]))


### 16d. What kind of `wrong_value` errors are left? (new)
`wrong_value` (right tools, wrong argument values) is most of the remaining error. Each wrong argument gets one label:

| Label | Meaning | Example |
|---|---|---|
| `copy_error` | the value was **in the question**, copied wrong | "Tokyo" → "Kyoto" |
| `number_not_in_text` | a number that had to be worked out | "two hundred and one" → 210 |
| `inferred_or_default` | a non-number that had to be inferred, or a default | `"year": ""` |
| `type_or_case_only` | right value, wrong type or case | `"8"` vs `8` |
| `left_out_arg` / `extra_arg` | an argument dropped / added | |

This tells DPO what to target: lots of `copy_error` → train on copying exactly; lots of `inferred_or_default` → the gold data's conventions (defaults) are the problem, not the model's reading.


In [ ]:
def value_errors(scored_df):
    rows = []
    for _, row in scored_df[scored_df["failure_type"] == "wrong_value"].iterrows():
        rec = records_by_id[row["id"]]
        pred = normalize_calls(extract_json(row["response"])) or []
        query, used = rec["query"].lower(), set()
        for g in rec["answers"]:
            cands = [(j, p) for j, p in enumerate(pred) if p["name"] == g["name"] and j not in used]
            if not cands:
                continue
            j, p = max(cands, key=lambda jp: sum(jp[1]["arguments"].get(k) == v for k, v in g["arguments"].items()))
            used.add(j)
            for k, v in g["arguments"].items():
                if k not in p["arguments"]:
                    kind = "left_out_arg"
                elif p["arguments"][k] == v:
                    continue
                elif str(p["arguments"][k]).lower() == str(v).lower():
                    kind = "type_or_case_only"
                elif str(v).lower() in query:
                    kind = "copy_error"
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    kind = "number_not_in_text"
                else:
                    kind = "inferred_or_default"
                rows.append(kind)
            rows += ["extra_arg" for k in p["arguments"] if k not in g["arguments"]]
    return pd.Series(rows, dtype=str)


value_error_mix = {}
for name, df in frames.items():
    errs = value_errors(df)
    value_error_mix[name] = {"wrong_value_questions": int((df["failure_type"] == "wrong_value").sum()),
                             "wrong_arguments": len(errs),
                             **(100 * errs.value_counts(normalize=True)).round(1).to_dict()}
display(pd.DataFrame(value_error_mix).fillna(0))


## 17. Save
Same files and `metrics.json` layout as the original run, plus the new sections: the comparison with the original SFT run, over-abstention, the 300-question irrelevance set, the chat judge, the `wrong_value` breakdown and the training summary. DPO will compare against `scored.csv` from here.


In [ ]:
sft_df.to_csv(RESULTS_DIR / "scored.csv", index=False)
train_summary_path = RESULTS_DIR / "train_summary.json"

metrics_report = {
    "run_name": RUN_NAME,
    "run_config": RUN_CONFIG,
    "n": len(sft_df),
    "summary": sft_summary.to_dict(orient="index"),
    "vs_base": comparison,
    "vs_original": vs_original,
    "over_abstention_pct": over_abstention,
    "breakdowns": {f: breakdown(sft_df, f).to_dict(orient="index") for f in FEATURES},
    "failure_mix": failure_mix(sft_df).to_dict(),
    "value_error_mix": value_error_mix,
    "generation": {
        "avg_gen_tokens": round(sft_df["gen_tokens"].mean(), 1),
        "hit_max_tokens_pct": round(100 * sft_df["hit_max_tokens"].mean(), 1),
        "latency_per_example_s": round(sft_df["latency_per_example"].mean(), 3),
        "peak_vram_gb": round(peak_eval_vram, 2) if peak_eval_vram else None,
    },
    "general_behaviour": {"chat_as_tool_call_pct": chat_tool_rate, "irrelevance_returns_empty_pct": irrelevance_acc},
    "irrelevance_v1": irrelevance,
    "chat_v2": {"tool_json_pct": chat_tool_json_pct, "judge_model": JUDGE_MODEL if chat_judge else None, "judge": chat_judge},
    "training": {
        "model": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "epochs": NUM_EPOCHS, "effective_batch": BATCH_SIZE * GRAD_ACCUM, "learning_rate": LEARNING_RATE,
        "abstain_fraction": ABSTAIN_FRACTION, "eval_subset": EVAL_SUBSET,
        "early_stopping": {"patience": EARLY_STOP_PATIENCE, "threshold": EARLY_STOP_THRESHOLD},
        "sft_v2_data": json.loads((SFT_V2_DIR / "sft_v2_report.json").read_text()),
        "summary": json.loads(train_summary_path.read_text()) if train_summary_path.exists() else None,
        "adapter_dir": str(ADAPTER_DIR), "merged_dir": str(MERGED_DIR),
    },
}
(RESULTS_DIR / "metrics.json").write_text(json.dumps(metrics_report, indent=2, default=str))
print(f"Saved to {RESULTS_DIR}")
if SMOKE_TEST:
    print("\n✅ Smoke test passed end-to-end. Set SMOKE_TEST = False and Run All for the real run.")
